In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib import transforms as mtransforms
from matplotlib.patches import Rectangle
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import scanpy as sc
import muon as mu
import infercnvpy as cnv
import io
import os
from pathlib import Path
from PIL import Image
from pypdf import PdfReader, PdfWriter, Transformation
import scipy.sparse as sp
import gc
import re


mpl.rcParams.update({
    # --- Fonts: keep text editable ---
    'pdf.fonttype': 'truetype',   # same as 42, ensures TrueType fonts
    'ps.fonttype': 'truetype',    # for EPS compatibility
    'svg.fonttype': 'none',       # keeps SVG text as text, not outlines

    # --- PDF structure: keep elements separate ---
    'pdf.compression': 0,         # prevents grouping/merging of vector paths
    'savefig.transparent': True,  # optional: preserves transparent backgrounds
    'savefig.bbox': 'tight',      # trims whitespace
    'savefig.pad_inches': 0.02,   # small padding for Illustrator bleed safety

    # --- General style niceties ---
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.family': 'sans-serif'
})

from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -20% !important; margin-right: -15% !important; }</style>"))

In [2]:
#Read in data from master h5mu file
adata = mu.read_h5ad('../Data/GEO_data_file_v1.h5mu', mod = 'rna')
print(adata.shape)

(38231, 28762)


In [3]:
adata.obs['timepoint'] = adata.obs['time']

In [4]:
#colors
colors_sheet = pd.read_excel('../data/production_color_v1.xlsx') #test sheet

#make leiden cluster color dictionary
colors_dict = dict(zip(colors_sheet.leiden, colors_sheet.colour))

In [5]:
#build some custom colour dictionaries for various parameters
#lclusters = ['2','6','17','21','5','16','12', '1','11','24','4','0','19','3', '10', '7','15','23','18',  '14', '8', '20', '22', '13','9', '25'] #ordered cluster list
lclusters = ['24',
 '11',
 '1',
 '12',
 '16',
 '5',
 '21',
 '17',
 '6',
 '2',
 '25',
 '9',
 '13',
 '22',
 '20',
 '8',
 '14',
 '18',
 '23',
 '15',
 '7',
 '10',
 '3',
 '19',
 '0',
 '4'] #modified ordered cluster list, keeps plots ordered as in barplots used in Fig3
lclusters_int = [int(i) for i in lclusters]
shared = [int(i) for i in lclusters[0:10]]
restricted = [int(i) for i in lclusters[10:]]
restricted_str = [i for i in lclusters[10:]]

patients = ['P01', 'P03', 'P11', 'P12', 'P09', 'P17', 'P18',  'P02', 'P08']

shared_cols = {}
restrict_cols = {}

for k in lclusters_int:
    if k in shared:
        shared_cols[k] = colors_dict[k]
        restrict_cols[k] = colors_dict['Atyp']
    elif k in restricted:
        shared_cols[k] = colors_dict['Atyp']
        restrict_cols[k] = colors_dict[k]        

#restrict_cols

In [6]:
#Get cnv data which is stored in  adata.uns
cnv_data = adata.uns['cnv']

In [7]:
# Create a list of autosomes to keep
chrom_to_keep = []
for i in range(23):
    chrom_to_keep.append('chr' + str(i))
    
# Read in a table of centromeres downloaded from UCSC on 2025_09_03
centromere = pd.read_csv('../Data/centromere_position.csv', sep = '\t')
centromere_positions = {}
for i in centromere['chrom'].unique():
    if i in chrom_to_keep:
        a = centromere.loc[centromere['chrom'] ==i, :]
        a = a.iloc[0, :]
        centromere_positions[i] = ((a['chromStart'] + a ['chromEnd'])/2).astype(int)

# Read in a table of centromeres length downloaded from UCSC on 2025_09_03
chrom_length = pd.read_csv('../Data/centromeres_banding.csv', sep = '\t')
chrom_length_dict = {}
for i in chrom_length['#chrom'].unique():
     if i in chrom_to_keep:
         a= chrom_length.loc[chrom_length['#chrom'] ==i, :]
         b = a['chromEnd'].max()
         chrom_length_dict[i] = b

FileNotFoundError: [Errno 2] No such file or directory: '../Data/centromeres_banding.csv'

In [ ]:
#Functions from HH to add pq annotations
# Create a function to describe the locations of each colorbar
def calculate_pq_positions_from_cnv_data(adata, heatmap_ax):
    """
    Calculate precise p/q positions using the CNV data structure from adata.
    
    Parameters:
    - adata: AnnData object with uns['cnv']['chr_pos'] structure
    - heatmap_ax: The heatmap axis from the plot
    """
    # Get chromosome positions from your data
    chr_pos = adata.uns['cnv']['chr_pos']
    
    # Sort chromosomes by their position in the matrix
    sorted_chrs = sorted(chr_pos.items(), key=lambda x: x[1])
    
    # Get the x-axis limits from the heatmap
    x_min, x_max = heatmap_ax.get_xlim()
    total_positions = adata.obsm['X_cnv'].shape[1]  # Total number of genomic positions
    
    pq_positions = []
    
    for i, (chr_name, start_idx) in enumerate(sorted_chrs):
        # Calculate end index for this chromosome
        if i < len(sorted_chrs) - 1:
            end_idx = sorted_chrs[i + 1][1] - 1
        else:
            end_idx = total_positions - 1
        
        # Calculate chromosome width in terms of genomic positions
        chr_width_positions = end_idx - start_idx + 1
        
        # Convert to plot coordinates
        chr_start_plot = x_min + (start_idx / total_positions) * (x_max - x_min)
        chr_end_plot = x_min + ((end_idx + 1) / total_positions) * (x_max - x_min)
        chr_width_plot = chr_end_plot - chr_start_plot
        
        # Calculate centromere position if we have the data
        if chr_name in centromere_positions and chr_name in chrom_length_dict:
            centromere_bp = centromere_positions[chr_name]
            chr_length_bp = chrom_length_dict[chr_name]
            
            # Calculate relative position of centromere (0 to 1)
            centromere_ratio = centromere_bp / chr_length_bp
            
            # Convert to plot coordinates
            centromere_plot = chr_start_plot + (chr_width_plot * centromere_ratio)
            
            # Calculate p and q arm centers
            p_center = (chr_start_plot + centromere_plot) / 2
            q_center = (centromere_plot + chr_end_plot) / 2
            
        else:
            # Fallback: assume centromere is roughly in the middle
            centromere_plot = (chr_start_plot + chr_end_plot) / 2
            p_center = (chr_start_plot + centromere_plot) / 2
            q_center = (centromere_plot + chr_end_plot) / 2
        
        pq_positions.append({
            'chromosome': chr_name,
            'start_idx': start_idx,
            'end_idx': end_idx,
            'chr_start_plot': chr_start_plot,
            'chr_end_plot': chr_end_plot,
            'centromere_plot': centromere_plot,
            'p_center': p_center,
            'q_center': q_center,
            'width_positions': chr_width_positions
        })
    
    return pq_positions

def add_pq_annotations_to_cnv_heatmap(fig, heatmap_ax, gene_groups_ax, adata, style='combined'):
    """
    Add p/q arm annotations to your CNV heatmap.
    
    Parameters:
    - fig: matplotlib figure
    - heatmap_ax: main heatmap axis
    - gene_groups_ax: chromosome groups axis (can be None)
    - adata: AnnData object with CNV data
    - style: 'labels', 'bars', 'combined', or 'separator_lines'
    """
    
    # Calculate positions
    pq_positions = calculate_pq_positions_from_cnv_data(adata, heatmap_ax)
    
    # Determine the position for annotations
    if gene_groups_ax is not None:
        gene_groups_pos = gene_groups_ax.get_position()
        base_y = gene_groups_pos.y1
        base_width = gene_groups_pos.width
        base_x0 = gene_groups_pos.x0
    else:
        heatmap_pos = heatmap_ax.get_position()
        base_y = heatmap_pos.y0 - 0.02
        base_width = heatmap_pos.width
        base_x0 = heatmap_pos.x0
    
    if style in ['labels', 'combined']:
        # Create annotation axis for p/q labels
        pq_label_ax = fig.add_axes([
            base_x0,
            base_y + 0.005,
            base_width,
            0.025
        ])
        
        pq_label_ax.set_xlim(heatmap_ax.get_xlim())

        
        # Clean up axis
        pq_label_ax.set_ylim(0, 1)
        pq_label_ax.set_xticks([])
        pq_label_ax.set_yticks([])
        for spine in pq_label_ax.spines.values():
            spine.set_visible(False)
    
    if style in ['bars', 'combined']:
        # Create colored bars for p/q arms
        bar_ax = fig.add_axes([
            base_x0,
            base_y + 0.035,
            base_width,
            0.012
        ])
        
        bar_ax.set_xlim(heatmap_ax.get_xlim())
        
        for pos_info in pq_positions:
            # P arm bar (blue)
            bar_ax.axhspan(0, 1, xmin=(pos_info['chr_start_plot'] - heatmap_ax.get_xlim()[0]) / (heatmap_ax.get_xlim()[1] - heatmap_ax.get_xlim()[0]),
                          xmax=(pos_info['centromere_plot'] - heatmap_ax.get_xlim()[0]) / (heatmap_ax.get_xlim()[1] - heatmap_ax.get_xlim()[0]),
                          color='grey', alpha=0.8, linewidth=0)
            
            # Q arm bar (red)
            bar_ax.axhspan(0, 1, xmin=(pos_info['centromere_plot'] - heatmap_ax.get_xlim()[0]) / (heatmap_ax.get_xlim()[1] - heatmap_ax.get_xlim()[0]),
                          xmax=(pos_info['chr_end_plot'] - heatmap_ax.get_xlim()[0]) / (heatmap_ax.get_xlim()[1] - heatmap_ax.get_xlim()[0]),
                          color='black', alpha=0.8, linewidth=0)
        
        # Clean up bar axis
        bar_ax.set_ylim(0, 1)
        bar_ax.set_xticks([])
        bar_ax.set_yticks([])
        for spine in bar_ax.spines.values():
            spine.set_visible(False)
    
    if style == 'separator_lines':
        # Add vertical lines to mark centromeres directly on the heatmap
        for pos_info in pq_positions:
            heatmap_ax.axvline(x=pos_info['centromere_plot'], color='white', linewidth=1, alpha=0.8)
            
            # Add small text labels on the heatmap
            y_pos = heatmap_ax.get_ylim()[1] - 0.02 * (heatmap_ax.get_ylim()[1] - heatmap_ax.get_ylim()[0])
            heatmap_ax.text(pos_info['p_center'], y_pos, 'p', 
                           ha='center', va='top', fontsize=6, fontweight='bold', color='white')
            heatmap_ax.text(pos_info['q_center'], y_pos, 'q', 
                           ha='center', va='top', fontsize=6, fontweight='bold', color='white')
    
    return pq_positions

In [ ]:
adata.obs.columns

In [ ]:
adata.obs['inferCNV_cluster'].drop_duplicates().to_list()

In [ ]:
#Need to convert inferCNV clusters back to old nomenclature to test here
icnv_dict = { 'P18_i':'0',
 'P18_iii':'2',
 'P18_iv':'3',
 'P18_ii':'1',
 'P18_vi':'5',
 'P18_ix':'8',
 'P18_v':'4',
 'P18_viii':'7',
 'P17_iv':'3',
 'P17_ii':'1',
 'P17_i':'0',
 'P17_v':'4',
 'P17_vii':'6',
 'P17_iii':'2',
 'P17_viii':'7',
 'P17_vi':'5',
 'P18_vii':'6',
 'P03_i':'0',
 'P03_ii':'1'}


In [ ]:
adata.obs['leiden_CNV'] = adata.obs['inferCNV_cluster'].replace(icnv_dict)
adata.obs['leiden_CNV'].drop_duplicates()

In [ ]:
# === CNV heatmap pipeline: combined/separate timepoints, patient-specific genotype,
#     right dendrogram, legend, chromosome p/q bar, and multi-key ordering ===

# ----------------------------
# USER CONFIG
# ----------------------------
group_key   = 'leiden'
patient_key = 'patient_alias'
time_key    = 'timepoint'

# Patients / timepoints / clusters to plot
patients    = ['P18', 'P17', 'P03']          # or your list
timepoints  = ['C1', 'C7', 'End']
restricted_only = False                      # or True to only plot patient-restricted clusters

# CNV source
cnv_source = 'auto'
cmap       = 'seismic'
vmin, vmax = -0.3, 0.3

# Row ordering / clustering mode: "within_leiden" | "global" | "none"
cluster_mode = "none"   # for pure categorical ordering
show_leiden_boundaries_in_global = False   # only used when boundary_key == 'leiden' AND cluster_mode == 'global'

# NEW: order rows by leiden_CNV + timepoint + leiden
order_by_leiden_cnv = True

# Patient-specific order for leiden_CNV clusters (top→bottom)
# Any extra clusters will be appended at the bottom automatically.
leiden_cnv_order_per_patient = {
    'P18': [8,7,4,1,6,3,2,5,0],
    'P17': [5,3,4,1,0,2,6,7],
    'P03': [1,0],
}

# Fixed order for timepoints (top→bottom)
timepoint_order = ['C1', 'C7', 'End']

# Dendrogram options
show_dendrogram     = False   # typically off when using pure categorical ordering
subcluster_cut_frac = 0.5     # only used in within_leiden mode
dendro_linewidth    = 0.6
dendro_color        = 'k'

# Legend options
show_legend             = True
legend_title_fontsize   = 9
legend_item_fontsize    = 8

# Plotting mode
combine_timepoints = True      # False = per-timepoint figs; True = one combined fig per patient

# Optional palettes for other annotations
geno_palette = {
    'wt': '#4997c9',
    'mt': '#c70e77',
    'unknown': '#ffffff',
    'na': '#ffffff'
}
time_palette = {'C1': '#D3D3D3', 'C7': '#A9A9A9', 'End': '#36454F'}

# Hard-wired palette for leiden_CNV (add 'na' explicitly)
leiden_cnv_palette = {
    str(0): '#4e79a7',
    str(1): '#f28e2b',
    str(2): '#e15759',
    str(3): '#76b7b2',
    str(4): '#59a14f',
    str(5): '#edc948',
    str(6): '#b07aa1',
    str(7): '#ff9da7',
    str(8): '#bab0ac',
    'na':  '#ffffff',   # missing / no-call
}

# Flags kept for API compatibility (p/q bar uses your helper)
show_chrom_labels     = True
show_chrom_boundaries = True
show_centromeres      = True

# Performance knobs
max_rows_preview = None         # e.g. 4000 for a quick preview; None = full
heat_w = 6.0                    # visual width for heatmap column

# Output dir
outdir = Path('../Figures')
outdir.mkdir(exist_ok=True, parents=True)

# Layout widths (tweak to taste)
heat_w        = 6.0
dendro_w_cfg  = 0.9   # dendrogram column width
legend_w_cfg  = 1.2   # legend column width
ann_w_each_cfg= 0.25  # per-annotation-strip width


# ----------------------------
# HELPERS
# ----------------------------
def _normalize_leiden_everywhere(adata, group_key, lclusters, colors_dict):
    """Normalize Leiden to strings across obs, lclusters, and palette."""
    obs_str = adata.obs[group_key].astype(str).str.strip()
    adata.obs[group_key] = obs_str
    lclusters_str = [str(x).strip() for x in lclusters]
    colors_by_str = {str(k).strip(): v for k, v in colors_dict.items()}
    adata.obs[group_key] = pd.Categorical(
        adata.obs[group_key],
        categories=lclusters_str,
        ordered=True
    )
    return lclusters_str, colors_by_str


def _get_cnv_matrix(adata, source='auto'):
    """Return dense float32 array for CNV plotting."""
    if source == 'auto':
        if 'cnv' in adata.layers:
            X = adata.layers['cnv']
        elif 'X_cnv' in adata.obsm:
            X = adata.obsm['X_cnv']
        elif 'cnv' in adata.obsm:
            X = adata.obsm['cnv']
        else:
            X = adata.X
    elif source == 'layer:cnv':
        X = adata.layers['cnv']
    elif source == 'obsm:X_cnv':
        X = adata.obsm['X_cnv']
    else:
        X = adata.X

    if sp.issparse(X):
        X = X.toarray()
    else:
        X = np.asarray(X)
    return X.astype(np.float32, copy=False)


def _hier_order(X, method='ward', metric='euclidean'):
    if X.shape[0] <= 2:
        return np.arange(X.shape[0])
    Z = linkage(X, method=method, metric=metric)
    leaves = dendrogram(Z, no_plot=True)['leaves']
    return np.asarray(leaves)


def _cat_to_rgb(series, palette, default="#cccccc"):
    """Map a categorical series to RGB using a palette (keys compared as stripped strings)."""
    to_rgb = mcolors.to_rgb
    vals = series.astype(str).str.strip()
    return np.array(
        [to_rgb(palette.get(v, default)) for v in vals],
        dtype=float
    )


def _row_order_and_tree(adata, group_key, lclusters_str, X, cluster_mode="within_leiden"):
    """
    Returns:
      row_order : np.ndarray of row indices
      Z_global  : linkage matrix if cluster_mode == 'global', else None
    """
    if cluster_mode == "none":
        values = adata.obs[group_key].astype(str).values
        idx_all = []
        for cat in lclusters_str:
            rows = np.where(values == cat)[0]
            if rows.size:
                idx_all.extend(rows.tolist())
        leftover = [i for i in range(adata.n_obs) if i not in idx_all]
        return np.array(idx_all + leftover, dtype=int), None

    if cluster_mode == "global":
        if X.shape[0] <= 2:
            leaves = np.arange(X.shape[0])
            return leaves, None
        Z = linkage(X, method='ward', metric='euclidean')
        leaves = dendrogram(Z, no_plot=True)['leaves']
        return np.asarray(leaves), Z

    # default: within_leiden
    values = adata.obs[group_key].astype(str).values
    idx_all = []
    for cat in lclusters_str:
        rows = np.where(values == cat)[0]
        if rows.size == 0:
            continue
        if rows.size > 2:
            ord_local = _hier_order(X[rows, :], method='ward', metric='euclidean')
            rows = rows[ord_local]
        idx_all.extend(rows.tolist())
    leftover = [i for i in range(adata.n_obs) if i not in idx_all]
    return np.array(idx_all + leftover, dtype=int), None


def _draw_block_dendrogram(axd, X_block, y_start, y_end, linecolor='k', lw=0.6):
    """Draw a dendrogram for rows [y_start, y_end) on axis axd (right of heatmap)."""
    n = X_block.shape[0]
    if n <= 2:
        return None, None
    Z = linkage(X_block, method='ward', metric='euclidean')
    dd = dendrogram(Z, orientation='right', no_plot=True)
    icoords = dd['icoord']
    dcoords = dd['dcoord']
    max_d = max([max(dc) for dc in dcoords]) if dcoords else 1.0
    if max_d == 0:
        max_d = 1.0

    def map_y(v): return y_start + (np.asarray(v) - 5.0) / 10.0
    def map_x(v): return np.asarray(v) / max_d

    for yy, xx in zip(icoords, dcoords):
        axd.plot(map_x(xx), map_y(yy), color=linecolor, lw=lw)
    return Z, max_d


def _within_block_subcluster_boundaries(Z, frac=0.7):
    """Cut linkage at frac*max_height; return local boundaries where labels change."""
    if Z is None or Z.shape[0] == 0 or frac is None:
        return np.array([], dtype=int)
    max_h = Z[:, 2].max() if Z.size else 0.0
    if max_h == 0:
        return np.array([], dtype=int)
    cut = float(frac) * float(max_h)
    labels = fcluster(Z, t=cut, criterion='distance')
    changes = np.flatnonzero(labels[:-1] != labels[1:]) + 1
    return changes


def _row_order_by_category(adata, cat_key, cat_order=None):
    """
    Pure categorical row ordering by one key.
    NaNs are treated as 'na'.
    """
    if cat_key not in adata.obs.columns:
        return np.arange(adata.n_obs)

    raw = adata.obs[cat_key].astype(object).values
    vals = []
    for x in raw:
        if pd.isna(x):
            vals.append('na')
        else:
            vals.append(str(x).strip())
    vals = np.array(vals, dtype=object)

    uniq = pd.unique(vals)
    if cat_order is None:
        ordered = [str(u) for u in uniq]
    else:
        base = [str(c) for c in cat_order]
        extras = [str(u) for u in uniq if str(u) not in base]
        ordered = base + extras

    idx_all = []
    for cat in ordered:
        rows = np.where(vals == cat)[0]
        if rows.size:
            idx_all.extend(rows.tolist())
    leftover = [i for i in range(adata.n_obs) if i not in idx_all]
    return np.array(idx_all + leftover, dtype=int)


def _row_order_by_multi_category(adata, keys, orders_dict=None):
    """
    Multi-key categorical ordering.

    keys: list like ['leiden_CNV', time_key, group_key]
    orders_dict: dict mapping key -> preferred order list (values will be cast to str).
                 Any categories not listed are appended at the end in the order they appear.

    Returns a numpy array of row indices (0..n_obs-1) giving the sorted order.
    """
    if orders_dict is None:
        orders_dict = {}

    df = pd.DataFrame(index=adata.obs.index)
    for k in keys:
        vals = adata.obs[k].astype(object)
        norm = vals.map(lambda x: 'na' if pd.isna(x) else str(x).strip())
        uniq = pd.unique(norm)
        base_order = orders_dict.get(k)

        if base_order is not None:
            base_order = [str(x) for x in base_order]
            cats = base_order + [x for x in uniq if x not in base_order]
        else:
            cats = [str(x) for x in uniq]

        df[k] = pd.Categorical(norm, categories=cats, ordered=True)

    # tie-breaker: original index
    df['_orig'] = np.arange(adata.n_obs)
    df_sorted = df.sort_values(keys + ['_orig'], kind='mergesort')
    return df_sorted['_orig'].to_numpy()


def _prepare_ann_strips(
    adata,
    row_order,
    lclusters_str,
    colors_by_str,
    genokey_for_patient=None,
    force_dummy_geno=False,
):
    """
    Build annotation strips (RGB columns) + keys list.

    Left→right order:
      1. leiden (group_key)
      2. leiden_CNV
      3. genotype
      4. timepoint
    """
    ann_keys = []
    ann_cols_rgb = []

    # 1) Leiden strip (group_key)
    if group_key in adata.obs.columns:
        vals = adata.obs[group_key].astype(str).str.strip().values[row_order]
        pal  = {cat: colors_by_str.get(cat, '#cccccc') for cat in lclusters_str}
        ann_keys.append(group_key)
        ann_cols_rgb.append(_cat_to_rgb(pd.Series(vals), pal))

    # 2) leiden_CNV strip (using hard-wired palette; map NaN → 'na')
    if 'leiden_CNV' in adata.obs.columns:
        vals_cnv = adata.obs['leiden_CNV'].astype(object).values[row_order]
        vals_cnv_norm = []
        for x in vals_cnv:
            if pd.isna(x):
                vals_cnv_norm.append('na')
            else:
                vals_cnv_norm.append(str(x).strip())
        ann_keys.append('leiden_CNV')
        ann_cols_rgb.append(_cat_to_rgb(pd.Series(vals_cnv_norm), leiden_cnv_palette))

    # 3) Genotype strip — real or dummy
    if genokey_for_patient and genokey_for_patient in adata.obs.columns:
        vals = (
            adata.obs[genokey_for_patient]
            .astype(str)
            .str.lower()
            .str.strip()
            .replace('nan', 'na')
            .fillna('na')
            .values[row_order]
        )
        ann_keys.append(genokey_for_patient)
        ann_cols_rgb.append(_cat_to_rgb(pd.Series(vals), geno_palette))
    elif force_dummy_geno:
        vals = pd.Series(['na'] * len(row_order))
        ann_keys.append('Genotype')
        ann_cols_rgb.append(_cat_to_rgb(vals, geno_palette))

    # 4) Timepoint strip
    if time_key in adata.obs.columns:
        vals_tp = adata.obs[time_key].astype(str).str.strip().values[row_order]
        ann_keys.append(time_key)
        ann_cols_rgb.append(_cat_to_rgb(pd.Series(vals_tp), time_palette))

    return ann_keys, ann_cols_rgb


def _legend_items_from_palette(palette, present_keys):
    """Return [(label, color), ...] filtered to 'present_keys' (preserving palette order)."""
    items = []
    if present_keys is None:
        for k, c in palette.items():
            items.append((str(k), c))
    else:
        for k in palette:
            if str(k) in present_keys:
                items.append((str(k), palette[k]))
    return items


def _draw_legend(ax, sections, title_fs=9, item_fs=8):
    """Draw stacked legend sections on 'ax' (vector)."""
    ax.set_axis_off()
    y = 1.0
    line_h = 0.025
    pad_t  = 0.03
    sw, sh = 0.06, 0.03  # swatch width/height in axes coords

    for sec in sections:
        title = sec.get('title', '')
        items = sec.get('items', [])
        if not items:
            continue

        y -= pad_t
        if title:
            ax.text(0.0, y, title, fontsize=title_fs,
                    fontweight='bold', va='top')
            y -= pad_t

        for lab, color in items:
            if y < 0.02:
                break
            rect = Rectangle(
                (0.0, y - sh),
                sw, sh,
                transform=ax.transAxes,
                facecolor=color,
                edgecolor='k',
                linewidth=0.3
            )
            ax.add_patch(rect)
            ax.text(
                sw + 0.02, y - sh/2, str(lab),
                fontsize=item_fs, va='center', ha='left'
            )
            y -= line_h
        y -= 0.02


# ----------------------------
# CORE PLOT (using your p/q info above the heatmap)
# ----------------------------
def plot_cnv_with_annotations(
    adata,
    cnv_source='auto',
    row_order=None,
    ann_keys=None, ann_cols_rgb=None,
    vmin=-0.3, vmax=0.3, cmap='seismic',
    title=None,
    downsample_rows_to=None,
    show_dendrogram=True,
    dendro_linewidth=0.6,
    dendro_color='k',
    subcluster_cut_frac=0.7,
    show_legend=True,
    legend_sections=None,
    chrom_axis=None,            # kept for API compatibility, now unused
    show_chrom_labels=True,     # unused
    show_chrom_boundaries=True, # unused
    show_centromeres=True,      # unused
    cluster_mode="within_leiden",
    Z_global=None,
    show_leiden_boundaries_in_global=show_leiden_boundaries_in_global,
    # NEW:
    boundary_key=None,
    show_boundaries=True,
):
    """
    Heatmap via imshow (raster) + annotation strips (vector) + right dendrogram (vector) + legend (vector).
    Chromosome p/q bars and vertical lines are drawn using your calculate_pq_positions_from_cnv_data helper.

    Boundaries (horizontal lines + labels) are drawn based on `boundary_key`.
    """
    X = _get_cnv_matrix(adata, cnv_source)
    if row_order is None:
        row_order = np.arange(adata.n_obs)
    X = X[row_order, :]

    # Optional downsample for speed
    if downsample_rows_to and X.shape[0] > downsample_rows_to:
        step = max(1, X.shape[0] // downsample_rows_to)
        X = X[::step, :]
        row_order = row_order[::step]
        if ann_cols_rgb is not None and len(ann_cols_rgb) > 0:
            ann_cols_rgb = [col[::step, :] for col in ann_cols_rgb]

    # Layout sizes
    nrows, ncols = X.shape
    n_anns = len(ann_keys) if ann_keys else 0
    dendro_w  = dendro_w_cfg   if show_dendrogram else 0.0
    legend_w  = legend_w_cfg   if show_legend else 0.0
    ann_w_each= ann_w_each_cfg if n_anns > 0 else 0.0

    fig_h = max(2.5, min(12.0, nrows / 120.0))
    fig_w = heat_w + dendro_w + legend_w + n_anns * ann_w_each + 1.0

    fig = plt.figure(figsize=(fig_w, fig_h))

    # --- Gridspec: [ann strips]*n_anns  +  [heatmap]  +  [dendrogram?]  +  [legend?]
    width_ratios = ([ann_w_each]*n_anns) + [heat_w]
    if show_dendrogram:
        width_ratios += [dendro_w]
    if show_legend:
        width_ratios += [legend_w]
    gs = fig.add_gridspec(1, len(width_ratios),
                          width_ratios=width_ratios, wspace=0.06)

    # Annotation strips (vector)
    if n_anns > 0:
        for i, (key, col) in enumerate(zip(ann_keys, ann_cols_rgb)):
            ax = fig.add_subplot(gs[0, i])
            ax.imshow(col.reshape(-1, 1, 3), aspect='auto',
                      interpolation='nearest', origin='upper')
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(key, fontsize=8, pad=2)

    # Heatmap (raster)
    heat_col = n_anns
    axh = fig.add_subplot(gs[0, heat_col])
    axh.imshow(
        X, aspect='auto', interpolation='nearest',
        cmap=cmap, vmin=vmin, vmax=vmax, origin='upper'
    )
    axh.set_xlim(0, X.shape[1])
    axh.set_ylim(X.shape[0], 0)
    axh.set_yticks([])
    axh.set_xlabel('CNV bins / genes')

    # Dendrogram axis (vector) on the RIGHT
    axd = None
    if show_dendrogram:
        axd = fig.add_subplot(gs[0, heat_col + 1])
        axd.set_ylim(nrows, 0)
        axd.set_xlim(0, 1.0)
        axd.set_xticks([]); axd.set_yticks([])
        axd.set_title('tree', fontsize=8, pad=2)

    # Legend axis (vector) far right
    axl = None
    if show_legend:
        idx = heat_col + (1 if show_dendrogram else 0) + 1
        axl = fig.add_subplot(gs[0, idx])
        if legend_sections:
            _draw_legend(axl, legend_sections,
                         title_fs=legend_title_fontsize,
                         item_fs=legend_item_fontsize)
        else:
            axl.set_axis_off()

    # --- Group_key-based block boundaries (for dendrogram only) ---
    block_starts = block_ends = None
    if group_key in adata.obs.columns:
        cats_g = pd.Categorical(adata.obs[group_key].astype(str).values[row_order])
        vals_g = np.asarray(cats_g)
        leiden_changes_g = (
            np.flatnonzero(vals_g[:-1] != vals_g[1:]) + 1
            if vals_g.size >= 2 else np.array([], int)
        )
        block_starts = np.r_[0, leiden_changes_g]
        block_ends   = np.r_[leiden_changes_g, vals_g.size]

    # --- Visible boundaries & labels based on boundary_key ---
    if boundary_key is not None and boundary_key in adata.obs.columns and show_boundaries:
        cats_b = pd.Categorical(adata.obs[boundary_key].astype(str).values[row_order])
        vals_b = np.asarray(cats_b)
        boundary_changes = (
            np.flatnonzero(vals_b[:-1] != vals_b[1:]) + 1
            if vals_b.size >= 2 else np.array([], int)
        )

        # honour the "hide Leiden boundaries in global mode" flag
        if not (boundary_key == group_key and
                cluster_mode == "global" and
                not show_leiden_boundaries_in_global):
            for y in boundary_changes:
                axh.axhline(y, color='k', lw=0.7, alpha=0.7)

            # labels at the left margin
            block_starts_b = np.r_[0, boundary_changes]
            block_ends_b   = np.r_[boundary_changes, vals_b.size]
            for b0, b1 in zip(block_starts_b, block_ends_b):
                label = str(vals_b[b0])
                if label.lower() != 'nan':
                    axh.text(-0.5, (b0 + b1)/2, label,
                             va='center', ha='right', fontsize=7)

    # --- Dendrogram (no dashed separators in GLOBAL mode) ---
    if show_dendrogram and axd is not None:
        if cluster_mode == "global":
            if Z_global is not None and Z_global.shape[0] > 0:
                dd = dendrogram(Z_global, orientation='right', no_plot=True)
                icoords, dcoords = dd['icoord'], dd['dcoord']
                max_d = max([max(dc) for dc in dcoords]) if dcoords else 1.0
                if max_d == 0:
                    max_d = 1.0

                def map_y(v): return (np.asarray(v) - 5.0) / 10.0
                def map_x(v): return np.asarray(v) / max_d

                for yy, xx in zip(icoords, dcoords):
                    axd.plot(map_x(xx), map_y(yy),
                             color=dendro_color, lw=dendro_linewidth)
            # (intentionally no dashed separators in global mode)

        else:
            # within_leiden: dendrogram per Leiden block + dashed within-block separators
            if block_starts is not None and block_ends is not None:
                for b0, b1 in zip(block_starts, block_ends):
                    if b1 - b0 <= 2:
                        continue
                    X_block = X[b0:b1, :]
                    Zb, _ = _draw_block_dendrogram(
                        axd, X_block, y_start=b0, y_end=b1,
                        linecolor=dendro_color, lw=dendro_linewidth
                    )
                    if subcluster_cut_frac is not None and Zb is not None:
                        local_changes = _within_block_subcluster_boundaries(
                            Zb, frac=subcluster_cut_frac
                        )
                        for dy in local_changes:
                            axh.axhline(b0 + dy, color='k',
                                        lw=0.5, alpha=0.45, linestyle='--')

    # --- Chromosome p/q bar ABOVE heatmap + vertical lines ---
    try:
        pq_positions = calculate_pq_positions_from_cnv_data(adata, axh)

        # bar axis above the heatmap
        heat_pos = axh.get_position()
        bar_height = 0.012
        gap = 0.01
        bar_ax = fig.add_axes([
            heat_pos.x0,
            heat_pos.y1 + gap,
            heat_pos.width,
            bar_height,
        ])
        bar_ax.set_xlim(axh.get_xlim())
        bar_ax.set_ylim(0, 1)
        bar_ax.set_xticks([]); bar_ax.set_yticks([])
        for spine in bar_ax.spines.values():
            spine.set_visible(False)

        x0, x1 = axh.get_xlim()
        span = x1 - x0 if x1 != x0 else 1.0

        for pos in pq_positions:
            # P arm bar (grey)
            bar_ax.axhspan(
                0, 1,
                xmin=(pos['chr_start_plot'] - x0) / span,
                xmax=(pos['centromere_plot'] - x0) / span,
                color='grey', alpha=0.8, linewidth=0
            )
            # Q arm bar (black)
            bar_ax.axhspan(
                0, 1,
                xmin=(pos['centromere_plot'] - x0) / span,
                xmax=(pos['chr_end_plot'] - x0) / span,
                color='black', alpha=0.8, linewidth=0
            )

        # vertical lines on the heatmap at boundaries and centromeres
        for pos in pq_positions:
            axh.axvline(pos['chr_start_plot'],   color='k',     lw=0.4, alpha=0.5)
            axh.axvline(pos['chr_end_plot'],     color='k',     lw=0.4, alpha=0.5)
            axh.axvline(pos['centromere_plot'],  color='white', lw=0.7,
                        alpha=0.9, linestyle='--')
    except Exception as e:
        print(f"Chromosome annotation failed: {e}")

    if title:
        fig.suptitle(title, y=1.02)

    fig.tight_layout()
    return fig


# ----------------------------
# MAIN: per-patient plotting
# ----------------------------
for p in patients:
    if restricted_only:
        cls = 'restricted_only'
        ad_p = adata[
            (adata.obs[patient_key] == p) &
            (adata.obs['leiden'].isin(restricted_str))
        ].copy()
    else:
        cls = 'all_clusters'
        ad_p = adata[adata.obs[patient_key] == p].copy()

    # Leiden normalization (full patient subset)
    lclusters_str, colors_by_str = _normalize_leiden_everywhere(
        ad_p, group_key, lclusters, colors_dict
    )

    # Patient-specific genotype column from mut_dict (optional)
    genokey_for_patient = mut_dict.get(p) if 'mut_dict' in globals() else None
    if genokey_for_patient and genokey_for_patient not in ad_p.obs.columns:
        genokey_for_patient = None  # skip silently
    use_dummy_geno = genokey_for_patient is None

    # Decide which key we use for visible boundaries
    # If ordering by leiden_CNV and it exists, use that for boundaries.
    # Otherwise, fall back to Leiden.
    if order_by_leiden_cnv and 'leiden_CNV' in ad_p.obs.columns:
        boundary_key_patient = 'leiden_CNV'
    else:
        boundary_key_patient = group_key

    if combine_timepoints:
        # --- COMBINED MODE ---
        if order_by_leiden_cnv and 'leiden_CNV' in ad_p.obs.columns:
            order_list_cnv = leiden_cnv_order_per_patient.get(p)
            orders = {
                'leiden_CNV': order_list_cnv,
                time_key: timepoint_order,
                group_key: lclusters_str,
            }
            row_order = _row_order_by_multi_category(
                ad_p,
                keys=['leiden_CNV', group_key, time_key],  #annotation order (change in 2 spots)
                orders_dict=orders
            )
            Z_global = None
        else:
            X_all = _get_cnv_matrix(ad_p, cnv_source)
            row_order, Z_global = _row_order_and_tree(
                ad_p, group_key, lclusters_str, X_all, cluster_mode=cluster_mode
            )

        # Annotation strips
        ann_keys, ann_cols_rgb = _prepare_ann_strips(
            ad_p, row_order, lclusters_str, colors_by_str,
            genokey_for_patient=genokey_for_patient,
            force_dummy_geno=use_dummy_geno
        )

        # Legend sections (filtered to PRESENT labels only)
        legend_sections = []

        # Leiden legend
        present_leiden = set(
            ad_p.obs[group_key].astype(str).str.strip().values[row_order]
        )
        leiden_items = _legend_items_from_palette(
            {cat: colors_by_str.get(cat, '#cccccc') for cat in lclusters_str},
            present_keys=present_leiden
        )
        if leiden_items:
            legend_sections.append({'title': 'Leiden', 'items': leiden_items})

        # Leiden_CNV legend
        if 'leiden_CNV' in ad_p.obs.columns:
            vals_cnv = ad_p.obs['leiden_CNV'].astype(object).values[row_order]
            present_cnv = set(
                'na' if pd.isna(x) else str(x).strip()
                for x in vals_cnv
            )
            cnv_items = _legend_items_from_palette(
                leiden_cnv_palette, present_keys=present_cnv
            )
            if cnv_items:
                legend_sections.append({'title': 'Leiden_CNV', 'items': cnv_items})

        # genotype legend (real or dummy)
        if genokey_for_patient or use_dummy_geno:
            if genokey_for_patient:
                present_geno = set(
                    ad_p.obs[genokey_for_patient]
                    .astype(str)
                    .str.lower()
                    .str.strip()
                    .replace('nan', 'na')
                    .fillna('na')
                    .values[row_order]
                )
                geno_title = genokey_for_patient
            else:
                present_geno = {'na'}
                geno_title = 'Genotype'

            geno_items = _legend_items_from_palette(
                geno_palette, present_keys=present_geno
            )
            if geno_items:
                legend_sections.append({'title': geno_title, 'items': geno_items})

        # timepoint legend
        present_tp = set(
            ad_p.obs[time_key].astype(str).str.strip().values[row_order]
        )
        time_items = _legend_items_from_palette(
            time_palette, present_keys=present_tp
        )
        if time_items:
            legend_sections.append({'title': 'Timepoint', 'items': time_items})

        fig = plot_cnv_with_annotations(
            ad_p,
            cnv_source=cnv_source,
            row_order=row_order,
            ann_keys=ann_keys, ann_cols_rgb=ann_cols_rgb,
            vmin=vmin, vmax=vmax, cmap=cmap,
            title=f'{p} — all timepoints (combined)',
            downsample_rows_to=max_rows_preview,
            show_dendrogram=show_dendrogram,
            dendro_linewidth=dendro_linewidth,
            dendro_color=dendro_color,
            subcluster_cut_frac=subcluster_cut_frac,
            show_legend=show_legend,
            legend_sections=legend_sections,
            chrom_axis=None,
            show_chrom_labels=show_chrom_labels,
            show_chrom_boundaries=show_chrom_boundaries,
            show_centromeres=show_centromeres,
            cluster_mode=cluster_mode,
            Z_global=Z_global,
            show_leiden_boundaries_in_global=show_leiden_boundaries_in_global,
            boundary_key=boundary_key_patient,
            show_boundaries=True,
        )
        out_pdf = outdir / f'5_{p}_COMBINED_custom_{cls}_v2_GEO.pdf'
        fig.savefig(out_pdf, format='pdf', dpi=300,
                    bbox_inches='tight', pad_inches=0.02)
        plt.show()
        plt.close(fig)
        del row_order
        if 'X_all' in locals():
            del X_all
        gc.collect()

    else:
        # --- SEPARATE MODE ---
        for tp in timepoints:
            ad_pt = ad_p[ad_p.obs[time_key] == tp].copy()
            if ad_pt.n_obs == 0:
                fig, ax = plt.subplots(figsize=(3, 3))
                ax.text(0.5, 0.5, f'{p} — {tp}\n(no data)',
                        ha='center', va='center')
                ax.axis('off')
                fp = outdir / f'5_{p}_{tp}_custom_{cls}_v4_GEO.pdf'
                fig.savefig(fp, format='pdf', dpi=300,
                            bbox_inches='tight', pad_inches=0.02)
                plt.show()
                plt.close(fig)
                continue

            # Normalize on the subset (categories may drop)
            lclusters_str, colors_by_str = _normalize_leiden_everywhere(
                ad_pt, group_key, lclusters_str, colors_by_str
            )

            # Decide boundary key for this subset
            if order_by_leiden_cnv and 'leiden_CNV' in ad_pt.obs.columns:
                boundary_key_tp = 'leiden_CNV'
            else:
                boundary_key_tp = group_key

            if order_by_leiden_cnv and 'leiden_CNV' in ad_pt.obs.columns:
                order_list_cnv = leiden_cnv_order_per_patient.get(p)
                orders = {
                    'leiden_CNV': order_list_cnv,
                    time_key: timepoint_order,
                    group_key: lclusters_str,
                }
                row_order = _row_order_by_multi_category(
                    ad_pt,
                    keys=['leiden_CNV', group_key, time_key],  #annotation order (change in 2 spots)
                    orders_dict=orders
                )
                Z_global = None
            else:
                X_pt = _get_cnv_matrix(ad_pt, cnv_source)
                row_order, Z_global = _row_order_and_tree(
                    ad_pt, group_key, lclusters_str, X_pt, cluster_mode=cluster_mode
                )

            # Annotation strips
            ann_keys, ann_cols_rgb = _prepare_ann_strips(
                ad_pt, row_order, lclusters_str, colors_by_str,
                genokey_for_patient=genokey_for_patient,
                force_dummy_geno=use_dummy_geno
            )

            # Legend sections (filtered to PRESENT labels only)
            legend_sections = []

            # Leiden legend
            present_leiden = set(
                ad_pt.obs[group_key].astype(str).str.strip().values[row_order]
            )
            leiden_items = _legend_items_from_palette(
                {cat: colors_by_str.get(cat, '#cccccc') for cat in lclusters_str},
                present_keys=present_leiden
            )
            if leiden_items:
                legend_sections.append({'title': 'Leiden', 'items': leiden_items})

            # Leiden_CNV legend
            if 'leiden_CNV' in ad_pt.obs.columns:
                vals_cnv = ad_pt.obs['leiden_CNV'].astype(object).values[row_order]
                present_cnv = set(
                    'na' if pd.isna(x) else str(x).strip()
                    for x in vals_cnv
                )
                cnv_items = _legend_items_from_palette(
                    leiden_cnv_palette, present_keys=present_cnv
                )
                if cnv_items:
                    legend_sections.append({'title': 'Leiden_CNV', 'items': cnv_items})

            # genotype legend (real or dummy)
            if genokey_for_patient or use_dummy_geno:
                if genokey_for_patient:
                    present_geno = set(
                        ad_pt.obs[genokey_for_patient]
                        .astype(str)
                        .str.lower()
                        .str.strip()
                        .replace('nan', 'na')
                        .fillna('na')
                        .values[row_order]
                    )
                    geno_title = genokey_for_patient
                else:
                    present_geno = {'na'}
                    geno_title = 'Genotype'

                geno_items = _legend_items_from_palette(
                    geno_palette, present_keys=presento_geno
                )
                if geno_items:
                    legend_sections.append({'title': geno_title, 'items': geno_items})

            # timepoint legend
            present_tp = set(
                ad_pt.obs[time_key].astype(str).str.strip().values[row_order]
            )
            time_items = _legend_items_from_palette(
                time_palette, present_keys=present_tp
            )
            if time_items:
                legend_sections.append({'title': 'Timepoint', 'items': time_items})

            fig = plot_cnv_with_annotations(
                ad_pt,
                cnv_source=cnv_source,
                row_order=row_order,
                ann_keys=ann_keys, ann_cols_rgb=ann_cols_rgb,
                vmin=vmin, vmax=vmax, cmap=cmap,
                title=f'{p} — {tp}',
                downsample_rows_to=max_rows_preview,
                show_dendrogram=show_dendrogram,
                dendro_linewidth=dendro_linewidth,
                dendro_color=dendro_color,
                subcluster_cut_frac=subcluster_cut_frac,
                show_legend=show_legend,
                legend_sections=legend_sections,
                chrom_axis=None,
                show_chrom_labels=show_chrom_labels,
                show_chrom_boundaries=show_chrom_boundaries,
                show_centromeres=show_centromeres,
                cluster_mode=cluster_mode,
                Z_global=Z_global,
                show_leiden_boundaries_in_global=show_leiden_boundaries_in_global,
                boundary_key=boundary_key_tp,
                show_boundaries=True,
            )
            out_pdf = outdir / f'5_{p}_{tp}_custom_{cls}_v2_GEO.pdf'
            fig.savefig(out_pdf, format='pdf', dpi=300,
                        bbox_inches='tight', pad_inches=0.02)
            plt.show()
            plt.close(fig)

            del row_order
            if 'X_pt' in locals():
                del X_pt
            gc.collect()

print("Done.")